# ViMedCSS — `vinai/PhoWhisper-large` vs `Reworkwhisper-large-v5`

Standalone notebook. Nothing from `D:\Fine_tune_wf` is uploaded: the two modules this
project's numbers depend on (`src/normalize.py`, `src/metrics.py`) are written to disk
verbatim by cells 5 and 6, so CER/WER here sit on exactly the same scale as every other
number in `docs/so-lieu-tong-hop.md`.

## Setup on Kaggle

1. New notebook, `File -> Import Notebook`, upload this file.
2. Settings: `Accelerator = GPU T4 x2` (one GPU is used), `Internet = On`.
3. Add-ons -> Secrets -> add `HF_TOKEN`. Required: `winhsss/Reworkwhisper-large-v5` is a
   private repo. ViMedCSS itself is CC BY 4.0 and not gated.
4. Run in order. Cell 3 is the only config cell.

## What it measures

`tensorxt/ViMedCSS` splits `test` (1,615 seg / 3.38 h) and `hard` (658 seg / 1.38 h).
Metric: **CER and WER**, corpus-level (sum of edits / sum of reference lengths).
Decode is plain greedy, `language="vi"`, `task="transcribe"`, no n-gram ban — the same
decode path as `src/asr.py`, so results are comparable with the project's other runs.

## Three gates

| Gate | Rule | If it fails |
|---|---|---|
| **G1** reproduce the paper | some normalization variant puts base PhoWhisper-large within ±1.0 pp CER of 19.25 and ±1.5 pp WER of 31.24 | do **not** compare against the paper at all; the base-vs-v5 head-to-head below is still valid, because both sides are normalized identically |
| **G2** v5 beats base | paired bootstrap CI of (CER_base − CER_v5) has lower bound > 0 | report INCONCLUSIVE, do not claim a win |
| **G3** SOTA claim | G1 and G2 both pass **and** CER(v5) < 19.25 | no SOTA claim |

Paper: `arxiv.org/pdf/2602.12911`, Table 4 (test, zero-shot): VietASR 27.56 WER / 20.38 CER ·
PhoWhisper-Large 31.24 / **19.25** · Whisper-Large-v3 34.47 / 24.61. The paper does not state
its scoring normalization, which is why G1 exists and why step 1 scores the **base** model,
not v5. **The hard-split baseline row has not been extracted from the PDF** — `PAPER["hard"]`
is deliberately empty in cell 3.

## Outputs (Kaggle output panel)

```
/kaggle/working/vimedcss/
  <model>.<split>.persegment.jsonl   raw, UNNORMALIZED hypotheses -- rescore without a GPU
  scores.json                        final CER/WER table + bootstrap intervals
  decode_failures.json               rows whose audio cannot be decoded, with the reason
  samples/<split>/*.wav              audio samples for listening, with ref/hyp printed inline
  vimedcss_output.zip
```

Audio is read from the column's **raw bytes with `soundfile`**, not through
`Audio(sampling_rate=...)`: that path decodes via torchcodec, which raises
`RuntimeError: No audio frames were decoded` on some rows of this dataset. Cell 9 decodes
every row up front; whatever fails is excluded from **both** models by `segment_id` and
counted, because dropping a row from one column only would make the two columns
incomparable.

In [ ]:
# Cell 2 -- dependencies. Probe before installing: Kaggle images already carry torch,
# transformers, datasets and soundfile, and a needless `pip install torch` has broken
# this project's images before.
import importlib, subprocess, sys

need = []
for mod, pkg in [("torch", "torch"), ("transformers", "transformers"),
                 ("datasets", "datasets"), ("soundfile", "soundfile"),
                 ("scipy", "scipy"), ("tqdm", "tqdm")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        need.append(pkg)

print("missing:", need or "nothing")
if need:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)

# peft is NOT needed: Reworkwhisper-large-v5 is a merged standalone checkpoint,
# not a LoRA adapter.

In [ ]:
# Cell 3 -- CONFIG. The only cell to edit.
import os
from pathlib import Path

# transformers' safetensors auto-conversion probe 403s on PhoWhisper-* (discussions
# disabled). Must be set before transformers is imported anywhere.
os.environ["TRANSFORMERS_AUTO_CONVERSION"] = "0"

DATASET_ID = "tensorxt/ViMedCSS"
BASE_MODEL = "vinai/PhoWhisper-large"
CAND_MODEL = "winhsss/Reworkwhisper-large-v5"      # private -> HF_TOKEN required
MODELS     = [BASE_MODEL, CAND_MODEL]
SPLITS     = ["test", "hard"]

# Paper Table 4, test split, zero-shot. hard split: NOT extracted from the PDF yet --
# leave empty rather than guess.
PAPER = {
    "test": {BASE_MODEL: {"wer": 31.24, "cer": 19.25}},
    "hard": {},
}
TOL_CER_PP = 1.0        # G1 tolerance; decode params are undisclosed, so an exact
TOL_WER_PP = 1.5        # match is not achievable

# Decode -- identical to src/asr.py: greedy, no n-gram ban.
BATCH_SIZE    = 8
NUM_BEAMS     = 1
LANGUAGE      = "vi"
MAX_CHUNK_SEC = 30.0    # Whisper's window. Longer segments are SPLIT, never truncated.
LIMIT         = None    # e.g. 20 for a smoke run of the whole notebook

# Scoring
FILLERS   = ["ừm", "ờm", "ehm", "uhm", "hmm"]   # ạ/à/ừ/ơ/dạ/vâng are lexical, never here
N_SAMPLES = 12          # audio samples exported per split for listening

OUT_DIR = Path("/kaggle/working/vimedcss") if Path("/kaggle/working").exists() else Path("Outputs/vimedcss")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("OUT_DIR:", OUT_DIR)

In [ ]:
# Cell 4 -- environment probe + HF token. Feature-detect, never branch on version
# strings: Kaggle patches images in place, so version numbers lie.
import sys, threading

def version_table():
    out = {"python": sys.version.split()[0]}
    for name in ("torch", "transformers", "datasets", "numpy", "scipy", "soundfile",
                 "huggingface_hub", "pyarrow"):
        try:
            out[name] = __import__(name).__version__
        except Exception as e:
            out[name] = f"<{type(e).__name__}>"
    return out

def silence_hf_discussions_403_noise():
    """The auto-conversion probe runs in a background thread and 403s on repos with
    discussions disabled; the exception prints from that thread, so try/except at the
    call site never sees it. Silenced at the thread-hook level (src/compat.py)."""
    default_hook = threading.excepthook
    def _hook(args):
        response = getattr(args.exc_value, "response", None)
        is_403 = response is not None and getattr(response, "status_code", None) == 403
        if is_403 and "discussions" in str(getattr(response, "url", "")):
            return
        default_hook(args)
    threading.excepthook = _hook

silence_hf_discussions_403_noise()

import torch
print(version_table())
print("cuda:", torch.cuda.is_available(),
      "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
      "| capability:", torch.cuda.get_device_capability(0) if torch.cuda.is_available() else "-")

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets")
except Exception as e:
    print(f"HF_TOKEN not loaded ({type(e).__name__}) -- {CAND_MODEL} is private and will 401")

In [ ]:
%%writefile vimed_normalize.py
"""Scoring normalization. PROJECT_CORE.md §6 Stage 4.

The only normalization function in the repo. Applied IDENTICALLY to hypothesis and
reference -- that symmetry is what makes the ambiguous cases safe: `một` is both "one"
and the indefinite article, `năm` is both "five" and "year", but if both sides convert
the same way the edit distance is unaffected. Never apply this to training targets.
"""

import re
import unicodedata
from dataclasses import dataclass, field

# Punctuation to drop. Keeps intra-word hyphen/apostrophe out of scope on purpose --
# Vietnamese does not use them and English tokens are kept verbatim.
_PUNCT = re.compile(r"[.,!?;:\"'`()\[\]{}<>/\\|~^*_=+&%$#@…“”‘’–—]")
_WS = re.compile(r"\s+")

_DIGITS = {
    "không": 0, "một": 1, "hai": 2, "ba": 3, "bốn": 4, "năm": 5,
    "sáu": 6, "bảy": 7, "tám": 8, "chín": 9,
    "tư": 4,    # positional variant: hai mươi tư = 24
    "lăm": 5,   # positional variant: mười lăm = 15
    "bẩy": 7,   # northern variant
}
_SCALES = {"nghìn": 1000, "ngàn": 1000, "triệu": 10**6, "tỷ": 10**9, "tỉ": 10**9}
_ZERO_FILLER = {"linh", "lẻ"}
_MULTIPLIERS = set(_SCALES) | _ZERO_FILLER | {"mười", "mươi", "trăm"}
_NUMWORDS = set(_DIGITS) | _MULTIPLIERS

# Vietnamese tone diacritics (sắc, huyền, hỏi, ngã, nặng) as NFD combining
# marks -- stripped before syllable-shape matching so tone doesn't matter,
# while the six extra vowel LETTERS (ă â ê ô ơ ư, not accents) are kept.
_TONE_MARKS = frozenset("̣́̀̉̃")

# Vietnamese syllable grammar: optional onset + required nucleus + optional coda.
# PROJECT_CORE.md §4's illustrative pattern only allows a single consonant
# letter and a single vowel letter, which flags ~99% of real Vietnamese words
# ("không", "được", "nhưng", ...) as non-Vietnamese -- unusable as literally
# written. This extends it with digraph onsets and diphthong/triphthong nuclei.
#
# Lives here rather than in scripts/inspect_errors.py (where it was written)
# because src/metrics.py needs it and inspect_errors imports from src.metrics --
# keeping it there would be a circular import.
_ONSETS = ["ngh", "ng", "nh", "ph", "th", "tr", "ch", "kh", "gi", "qu", "gh",
           "b", "c", "d", "đ", "g", "h", "k", "l", "m", "n", "p", "q", "r",
           "s", "t", "v", "x", "y"]
_NUCLEI = ["oai", "oay", "uao", "uay", "uôi", "ươi", "ươu", "iêu", "yêu",
           "uyu", "uya", "oeo", "uyê",
           "ia", "ya", "iê", "yê", "ua", "uô", "ưa", "ươ", "oa", "oe", "uy",
           "uơ", "uâ", "oă", "uê",
           "ai", "ay", "ây", "ao", "au", "âu", "eo", "êu", "oi", "ôi", "ơi",
           "ui", "ưi", "iu", "ưu",
           "a", "ă", "â", "e", "ê", "i", "o", "ô", "ơ", "u", "ư", "y"]
_CODAS = ["ng", "nh", "ch", "c", "m", "n", "p", "t"]


def _alternation(options: list[str]) -> str:
    return "|".join(sorted(options, key=len, reverse=True))


_SYLLABLE = re.compile(
    f"^(?:{_alternation(_ONSETS)})?(?:{_alternation(_NUCLEI)})(?:{_alternation(_CODAS)})?$"
)


def strip_tone(word: str) -> str:
    decomposed = unicodedata.normalize("NFD", word)
    return unicodedata.normalize("NFC", "".join(c for c in decomposed if c not in _TONE_MARKS))


def is_vietnamese_shaped(token: str) -> bool:
    return bool(_SYLLABLE.fullmatch(strip_tone(token.lower())))


def _parse_run(tokens: list[str]) -> str:
    """Vietnamese number words -> digits.

    A run with no multiplier is read as a DIGIT STRING, the way phone numbers and codes
    are spoken: "không chín tám bảy" -> "0987", not 0+9+8+7. Otherwise arithmetic, where
    `mười` is standalone ten, `mươi` a tens multiplier, and `linh`/`lẻ` a zero
    placeholder (một trăm linh năm = 105).
    """
    if len(tokens) > 1 and not any(t in _MULTIPLIERS for t in tokens):
        return "".join(str(_DIGITS[t]) for t in tokens)
    return str(_arith(tokens))


def _arith(tokens: list[str]) -> int:
    total = 0     # groups already multiplied by a scale (nghìn and up)
    cur = 0       # current group, < 1000
    pending = 0   # bare digit(s) awaiting a multiplier
    prev_digit = False
    for t in tokens:
        if t == "mười":
            cur += 10
            pending = 0
        elif t == "mươi":
            cur += pending * 10
            pending = 0
        elif t == "trăm":
            cur += pending * 100
            pending = 0
        elif t in _SCALES:
            total += (cur + pending) * _SCALES[t]
            cur = pending = 0
        elif t in _ZERO_FILLER:
            continue
        else:
            # Consecutive bare digits accumulate as a written number, not as a
            # replacement: "hai ba nghìn" is 23 nghìn, not 3 nghìn.
            pending = pending * 10 + _DIGITS[t] if prev_digit else _DIGITS[t]
        prev_digit = t not in _MULTIPLIERS
    return total + cur + pending


def words_to_digits(text: str) -> tuple[str, int]:
    """Collapse maximal runs of number words into digits. Returns (text, n_conversions).

    A lone `không` is left alone -- it is overwhelmingly the negation particle, and a
    spoken zero only ever appears inside a longer run (phone numbers, codes).
    """
    toks = text.split()
    out, n, i = [], 0, 0
    while i < len(toks):
        if toks[i] in _NUMWORDS:
            j = i
            while j < len(toks) and toks[j] in _NUMWORDS:
                j += 1
            run = toks[i:j]
            # strip trailing filler-only words so "năm linh" does not eat the linh --
            # only while more than one token remains, so a lone filler word (e.g. the
            # name "Linh") falls through to the `skip` branch below instead of being
            # stripped down to an empty run, which left `i` stuck at the same index
            # forever (infinite loop, hit on real Kaggle data 2026-08-01).
            while len(run) > 1 and run[-1] in _ZERO_FILLER:
                j -= 1
                run = run[:-1]
            skip = len(run) == 1 and run[0] in _ZERO_FILLER | {"không"}
            if run and not skip:
                out.append(_parse_run(run))
                n += 1
            else:
                out.extend(run)
            i = j
        else:
            out.append(toks[i])
            i += 1
    return " ".join(out), n


@dataclass
class NormStats:
    """Audit counters. `conversions` gating on 0 catches a silently broken parser."""
    conversions: int = 0
    fillers_removed: int = 0


@dataclass
class Normalizer:
    strip_punctuation: bool = True
    lowercase: bool = True
    number_convention: str = "word_to_digit"   # or "as_written"
    filler_tokens: list[str] = field(default_factory=list)
    stats: NormStats = field(default_factory=NormStats)

    def __post_init__(self):
        if self.number_convention not in ("word_to_digit", "as_written"):
            raise ValueError(f"unknown number_convention: {self.number_convention}")
        self._fillers = {unicodedata.normalize("NFC", f.lower()) for f in self.filler_tokens}

    def __call__(self, text: str) -> str:
        t = unicodedata.normalize("NFC", text)
        if self.lowercase:
            t = t.lower()
        if self.strip_punctuation:
            t = _PUNCT.sub(" ", t)
        t = _WS.sub(" ", t).strip()
        if self._fillers:
            toks = [x for x in t.split() if x not in self._fillers]
            self.stats.fillers_removed += len(t.split()) - len(toks)
            t = " ".join(toks)
        if self.number_convention == "word_to_digit":
            t, n = words_to_digits(t)
            self.stats.conversions += n
        return t

In [ ]:
%%writefile vimed_metrics.py
"""CER / WER and confidence intervals. No jiwer -- one less version to fight.

CER is the headline metric for Vietnamese ASR. Always CORPUS-level
(sum of edits / sum of reference lengths), never the mean of per-segment CERs: the
latter lets a 3-character segment outweigh a 300-character one.
"""

import random
from dataclasses import dataclass


def levenshtein(a, b) -> int:
    """Edit distance over any two sequences. Two rows of memory, O(len(a)*len(b)) time."""
    if a == b:
        return 0
    if len(a) < len(b):
        a, b = b, a
    if not b:
        return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1,          # deletion
                           cur[j - 1] + 1,       # insertion
                           prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]


@dataclass
class Counts:
    """Per-segment edit counts, kept unaggregated so bootstrap can resample them."""
    edits: int
    ref_len: int


def char_counts(ref: str, hyp: str) -> Counts:
    return Counts(levenshtein(ref, hyp), len(ref))


def word_counts(ref: str, hyp: str) -> Counts:
    r = ref.split()
    return Counts(levenshtein(r, hyp.split()), len(r))


def rate(counts: list[Counts]) -> float:
    """Corpus-level error rate. Empty reference corpus is a caller bug, not a 0.0."""
    denom = sum(c.ref_len for c in counts)
    if denom == 0:
        raise ValueError("reference corpus has zero length -- nothing to score")
    return sum(c.edits for c in counts) / denom


def score(refs: list[str], hyps: list[str]) -> dict:
    """CER and WER over aligned reference/hypothesis lists, plus the raw per-segment
    counts a gate needs for its interval."""
    if len(refs) != len(hyps):
        raise ValueError(f"length mismatch: {len(refs)} refs vs {len(hyps)} hyps")
    cc = [char_counts(r, h) for r, h in zip(refs, hyps)]
    wc = [word_counts(r, h) for r, h in zip(refs, hyps)]
    return {
        "n_segments": len(refs),
        "cer": rate(cc),
        "wer": rate(wc),
        "char_edits": sum(c.edits for c in cc),
        "char_ref_len": sum(c.ref_len for c in cc),
        "_char_counts": cc,
    }


def english_token_retention(refs: list[str], hyps: list[str]) -> dict:
    """Share of the reference's non-Vietnamese-shaped tokens the hypothesis
    reproduces verbatim, corpus-level (sum retained / sum candidates).

    CER cannot substitute for this. Two reasons, both measured on the
    v3-r16 -> v4-mixed-r16 regression this was written for:

      * A loanword is a handful of characters in a segment of hundreds, so
        losing every one of them moves CER by a fraction of a point -- inside
        the bootstrap interval, indistinguishable from noise.
      * Substituting a Vietnamese-shaped homophone (`team` -> `tim`,
        `build` -> `bill`) costs 2 edit characters while destroying the token
        for anything downstream that reads entities out of the transcript.

    Candidate selection is the same filter scripts/plot_youtube_stats.py and
    scripts/probe_youtube_captions.py already use -- `len > 1`, `isalpha()`,
    and failing the Vietnamese syllable-shape test -- so the three numbers stay
    comparable. `isalpha()` drops digits and alphanumerics; the syllable test
    rather than an English whitelist because a whitelist measured a 72%
    false-positive rate (CLAUDE.md).

    Matching is multiset membership over the whole hypothesis segment, not
    positional alignment: a token moved within the segment is still recognised,
    but a token said twice and transcribed once counts as one retained. Both
    sides must already be normalized the same way -- casing survives
    normalization nowhere in this pipeline, so this measures SPELLING only
    (`tim` for `team`), never casing.
    """
    if len(refs) != len(hyps):
        raise ValueError(f"length mismatch: {len(refs)} refs vs {len(hyps)} hyps")

    from collections import Counter

    from vimed_normalize import is_vietnamese_shaped

    n_candidates = n_retained = 0
    missing: Counter = Counter()
    for ref, hyp in zip(refs, hyps):
        cands = Counter(t for t in ref.split()
                        if len(t) > 1 and t.isalpha() and not is_vietnamese_shaped(t))
        if not cands:
            continue
        present = Counter(hyp.split())
        for token, want in cands.items():
            got = min(want, present[token])
            n_candidates += want
            n_retained += got
            if got < want:
                missing[token] += want - got
    return {
        "retention": n_retained / n_candidates if n_candidates else None,
        "n_candidates": n_candidates,
        "n_retained": n_retained,
        "missing": dict(missing.most_common()),
    }


def bootstrap_ci(counts: list[Counts], n_resamples: int = 1000, seed: int = 42,
                 alpha: float = 0.05) -> tuple[float, float]:
    """Segment-level bootstrap CI for one corpus rate."""
    rng = random.Random(seed)
    k = len(counts)
    vals = []
    for _ in range(n_resamples):
        pick = [counts[rng.randrange(k)] for _ in range(k)]
        denom = sum(c.ref_len for c in pick)
        if denom:
            vals.append(sum(c.edits for c in pick) / denom)
    vals.sort()
    lo = vals[int(alpha / 2 * len(vals))]
    hi = vals[min(len(vals) - 1, int((1 - alpha / 2) * len(vals)))]
    return lo, hi


def bootstrap_delta_ci(base: list[Counts], cand: list[Counts], n_resamples: int = 1000,
                       seed: int = 42, alpha: float = 0.05) -> tuple[float, float]:
    """PAIRED bootstrap CI for (base_cer - cand_cer): positive means cand is better.

    Both lists must be the same segments in the same order -- resampling them
    independently would discard the pairing and inflate the interval.
    """
    if len(base) != len(cand):
        raise ValueError("paired bootstrap needs the same segments on both sides")
    rng = random.Random(seed)
    k = len(base)
    vals = []
    for _ in range(n_resamples):
        idx = [rng.randrange(k) for _ in range(k)]
        bd = sum(base[i].ref_len for i in idx)
        cd = sum(cand[i].ref_len for i in idx)
        if bd and cd:
            vals.append(sum(base[i].edits for i in idx) / bd
                        - sum(cand[i].edits for i in idx) / cd)
    vals.sort()
    lo = vals[int(alpha / 2 * len(vals))]
    hi = vals[min(len(vals) - 1, int((1 - alpha / 2) * len(vals)))]
    return lo, hi


def verdict(lo: float, hi: float) -> str:
    """A CI straddling zero is INCONCLUSIVE, not a pass. With 196 real-bench segments
    the interval is ~±1.5pp, so smaller differences genuinely cannot be resolved."""
    if lo > 0:
        return "IMPROVED"
    if hi < 0:
        return "REGRESSED"
    return "INCONCLUSIVE"

In [ ]:
# Cell 7 -- import the two written modules and build the normalization variants.
from vimed_normalize import Normalizer, is_vietnamese_shaped
from vimed_metrics import (score, char_counts, word_counts, rate,
                           bootstrap_ci, bootstrap_delta_ci, verdict,
                           english_token_retention)

VARIANTS = {
    "N0 raw":          dict(strip_punctuation=False, lowercase=False, number_convention="as_written",    filler_tokens=[]),
    "N1 lower+punct":  dict(strip_punctuation=True,  lowercase=True,  number_convention="as_written",    filler_tokens=[]),
    "N2 +digits":      dict(strip_punctuation=True,  lowercase=True,  number_convention="word_to_digit", filler_tokens=[]),
    "N3 repo default": dict(strip_punctuation=True,  lowercase=True,  number_convention="word_to_digit", filler_tokens=FILLERS),
}

def norm_pair(refs, hyps, variant):
    """A fresh Normalizer per side: `stats` is mutable state, so sharing one instance
    would mix the two sides' conversion counters."""
    kw = VARIANTS[variant]
    nr, nh = Normalizer(**kw), Normalizer(**kw)
    return [nr(r) for r in refs], [nh(h) for h in hyps], nr.stats, nh.stats

# Set by the G1 cell once a variant reproduces the paper's base row. Every later cell
# reads it, so a failed G1 leaves it at the repo default and says so out loud.
SCORING_VARIANT = "N3 repo default"
print("variants:", list(VARIANTS))

In [ ]:
# Cell 8 -- load ViMedCSS and PROBE it. No GPU. This cell answers the questions that
# decide whether the decode below is sound; do not skip it.
from datasets import load_dataset, Audio
import numpy as np

DS = {}
for split in SPLITS:
    d = load_dataset(DATASET_ID, split=split)
    if LIMIT:
        d = d.select(range(min(LIMIT, len(d))))
    # decode=False on purpose: Audio(sampling_rate=...) routes decoding through
    # torchcodec, which raises `RuntimeError: No audio frames were decoded` on some
    # rows of this dataset (hit on Kaggle 2026-09-09). Cell 9 decodes the raw bytes
    # with soundfile instead.
    DS[split] = d.cast_column("audio", Audio(decode=False))

for split, d in DS.items():
    dur = np.asarray(d["duration_seconds"], dtype="float64")
    over = int((dur > MAX_CHUNK_SEC).sum())
    print(f"\n=== {split}: {len(d)} segments | {dur.sum()/3600:.2f} h")
    print(f"    duration s: min {dur.min():.1f} | median {np.median(dur):.1f} | "
          f"p95 {np.percentile(dur,95):.1f} | max {dur.max():.1f}")
    print(f"    longer than {MAX_CHUNK_SEC:.0f}s: {over} ({over/len(d)*100:.2f}%) "
          f"-> split into chunks, not truncated")
    print("    columns:", d.column_names)
    row = d[0]
    print("    segment_id:   ", row["segment_id"])
    print("    segment_text: ", row["segment_text"][:200])
    print("    cs_terms_list:", repr(row["cs_terms_list"])[:200], "| count:", row["cs_terms_count"])
    a = row["audio"]
    print("    audio obj:    ", type(a).__name__,
          "| keys:", list(a) if isinstance(a, dict) else "-",
          "| path:", (a.get("path") if isinstance(a, dict) else None),
          "| bytes:", (len(a["bytes"]) if isinstance(a, dict) and a.get("bytes") else None))

In [ ]:
# Cell 9 -- audio reader, then a decode check over EVERY row.
#
# soundfile, not torchcodec: `Audio(sampling_rate=...)` decodes via torchcodec and dies
# with `RuntimeError: No audio frames were decoded` on some rows of this dataset. Raw
# bytes -> soundfile -> scipy resample is the path src/data.py already uses, and it keeps
# one fewer library in the loop. The other shapes are kept as fallbacks in case a
# different `datasets` version hands back a decoded column.
import io, json
from math import gcd

import soundfile as sf
from scipy.signal import resample_poly
from tqdm.auto import tqdm

TARGET_SR = 16000

def _decode(raw, path):
    """soundfile first; torchcodec on the bytes as a fallback. The torchcodec failure
    that motivated this file was in `Audio(sampling_rate=...)`'s resampling path, so
    decoding the raw bytes with it may still work where libsndfile lacks the codec."""
    errs = []
    if raw:
        try:
            return sf.read(io.BytesIO(raw), dtype="float32", always_2d=False)
        except Exception as e:
            errs.append(f"soundfile: {type(e).__name__}")
        try:
            from torchcodec.decoders import AudioDecoder
            s = AudioDecoder(raw).get_all_samples()
            return s.data.cpu().numpy(), int(s.sample_rate)
        except Exception as e:
            errs.append(f"torchcodec: {type(e).__name__}")
    if path:
        try:
            return sf.read(path, dtype="float32", always_2d=False)
        except Exception as e:
            errs.append(f"soundfile(path): {type(e).__name__}")
    raise ValueError("; ".join(errs) or "row has neither bytes nor path")

def read_16k_mono(a) -> np.ndarray:
    if isinstance(a, dict) and ("bytes" in a or "path" in a):      # decode=False
        arr, sr = _decode(a.get("bytes"), a.get("path"))
    elif isinstance(a, dict):                                      # decoded column
        arr, sr = a["array"], a["sampling_rate"]
    elif hasattr(a, "get_all_samples"):                            # torchcodec
        s = a.get_all_samples()
        arr, sr = s.data.cpu().numpy(), int(s.sample_rate)
    else:
        raise TypeError(f"unhandled audio object: {type(a)}")

    arr = np.asarray(arr, dtype="float32")
    if arr.ndim > 1:                                     # (channels, n) or (n, channels)
        arr = arr.mean(axis=0) if arr.shape[0] < arr.shape[-1] else arr.mean(axis=-1)
    if arr.size == 0:
        raise ValueError("decoded 0 samples")
    if sr != TARGET_SR:
        g = gcd(int(sr), TARGET_SR)
        arr = resample_poly(arr, TARGET_SR // g, int(sr) // g).astype("float32")
    return arr

def chunk_audio(arr: np.ndarray) -> list:
    """Whisper reads a 30 s window and silently DROPS everything past it. A fixed
    boundary can cut a word, which costs a few characters -- truncation costs the whole
    tail, so splitting is the lesser error. Returns [arr] for normal segments."""
    w = int(MAX_CHUNK_SEC * TARGET_SR)
    if len(arr) <= w:
        return [arr]
    return [arr[i:i + w] for i in range(0, len(arr), w)]

# Decode every row now, before any GPU time. A row that cannot be decoded is excluded
# from BOTH models by segment_id -- excluding it from only one would make the two
# columns of the final table incomparable. The count is printed and stored.
BAD = {}
for split in SPLITS:
    d, bad = DS[split], {}
    for i in tqdm(range(len(d)), desc=f"decode-check {split}", unit="seg"):
        row = d[i]
        try:
            arr = read_16k_mono(row["audio"])
        except Exception as e:
            bad[row["segment_id"]] = f"{type(e).__name__}: {e}"
            continue
        got, want = len(arr) / TARGET_SR, row["duration_seconds"]
        if want and abs(got - want) > max(1.0, 0.1 * want):
            bad[row["segment_id"]] = f"length mismatch: decoded {got:.1f}s vs manifest {want:.1f}s"
    BAD[split] = bad
    print(f"{split}: {len(bad)} of {len(d)} rows unusable ({len(bad)/len(d)*100:.2f}%)")
    for sid, why in list(bad.items())[:5]:
        print("   ", sid, "->", why)

(OUT_DIR / "decode_failures.json").write_text(
    json.dumps(BAD, ensure_ascii=False, indent=2), encoding="utf-8")

In [ ]:
# Cell 10 -- model loading + decode, ported from src/asr.py.
import gc, json, re, subprocess, sys
from tqdm.auto import tqdm

def pick_dtype(device_index: int = 0):
    """fp16 vs bf16 by compute capability, never by torch.cuda.is_bf16_supported():
    that call returns True on T4 (sm_75), which has no native bf16 -- it lies."""
    if not torch.cuda.is_available():
        return torch.float32
    major, _ = torch.cuda.get_device_capability(device_index)
    return torch.float16 if major < 8 else torch.bfloat16

def repo_files(model_id: str) -> list:
    """What the repo actually contains. The load strategy is read off this instead of
    assumed: src/asr.py passes `use_safetensors=False` (vinai/PhoWhisper-large ships
    pytorch_model.bin), and that flag makes a safetensors-only repo fail with
    `does not appear to have a file named pytorch_model.bin or model.safetensors` --
    how Reworkwhisper-large-v5 failed here on 2026-09-09. The flag is not needed: the
    auto-conversion probe it was dodging is already handled by
    TRANSFORMERS_AUTO_CONVERSION=0 plus the thread hook in cell 4."""
    try:
        from huggingface_hub import list_repo_files
        return list_repo_files(model_id, token=os.environ.get("HF_TOKEN"))
    except Exception as e:
        print(f"  cannot list {model_id}: {type(e).__name__}: {e}")
        return []

def load_for_eval(model_id: str):
    from transformers import WhisperForConditionalGeneration, WhisperProcessor
    dtype = pick_dtype()
    files = repo_files(model_id)
    is_adapter = "adapter_config.json" in files
    weights = [f for f in files if f.endswith((".safetensors", ".bin"))]
    print(f"  {model_id}: {'LoRA adapter' if is_adapter else 'full checkpoint'} | "
          f"weights: {weights[:4]}{' ...' if len(weights) > 4 else ''}")

    # An adapter repo needs the base weights plus peft; a merged checkpoint needs neither.
    src = BASE_MODEL if is_adapter else model_id
    try:
        model = WhisperForConditionalGeneration.from_pretrained(src, torch_dtype=dtype)
    except OSError as e:
        print(f"  default load failed ({e}); retrying with use_safetensors=False")
        model = WhisperForConditionalGeneration.from_pretrained(
            src, torch_dtype=dtype, use_safetensors=False)

    if is_adapter:
        try:
            from peft import PeftModel
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft"], check=True)
            from peft import PeftModel
        model = PeftModel.from_pretrained(model, model_id)

    try:
        processor = WhisperProcessor.from_pretrained(model_id)
    except Exception as e:
        # A merge script that pushed weights but no tokenizer/feature-extractor files.
        # The vocab is the base model's either way, so this changes nothing measured.
        print(f"  no processor in {model_id} ({type(e).__name__}) -> using {BASE_MODEL}'s")
        processor = WhisperProcessor.from_pretrained(BASE_MODEL)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device).eval()
    for p in model.parameters():
        p.requires_grad_(False)
    return model, processor

def transcribe(model, processor, audios: list) -> list:
    """language=/task=, not forced_decoder_ids: the latter is gone from generate()'s
    signature in transformers 5.x and raises there, while language/task work on both."""
    device = next(model.parameters()).device
    dtype = next(model.parameters()).dtype
    inputs = processor(audios, sampling_rate=TARGET_SR, return_tensors="pt")
    features = inputs.input_features.to(device=device, dtype=dtype)
    with torch.no_grad():
        ids = model.generate(input_features=features, language=LANGUAGE,
                             task="transcribe", num_beams=NUM_BEAMS)
    return processor.batch_decode(ids, skip_special_tokens=True)

def slug(model_id: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", model_id.lower()).strip("_")

def jsonl_path(model_id: str, split: str) -> Path:
    return OUT_DIR / f"{slug(model_id)}.{split}.persegment.jsonl"

def free(*objs):
    for o in objs:
        del o
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Check both repos are reachable and see what shape they are BEFORE the decode loop:
# a 401 or a missing weight file costs nothing here and an hour of decoding later.
for _m in MODELS:
    _f = repo_files(_m)
    print(f"{_m}: {len(_f)} files"
          + ("  <-- empty: HF_TOKEN missing, or repo unreachable" if not _f else ""))
    print("   ", [x for x in _f if x.endswith((".json", ".safetensors", ".bin", ".txt"))][:12])

In [ ]:
# Cell 11 -- decode one (model, split) to JSONL. Hypotheses are stored RAW: every
# normalization decision below is made on these files, with no GPU.
#
# Written line-by-line and flushed per batch, and resumable by segment_id -- a Kaggle
# session that dies at 80% costs 20%, not the whole run.
def decode_split(model_id: str, split: str, force: bool = False) -> Path:
    d = DS[split]
    out = jsonl_path(model_id, split)
    done = set()
    if out.exists() and not force:
        with out.open(encoding="utf-8") as f:
            done = {json.loads(line)["segment_id"] for line in f if line.strip()}
    bad = BAD[split]
    todo = [i for i, sid in enumerate(d["segment_id"]) if sid not in done and sid not in bad]
    print(f"{model_id} / {split}: {len(done)} done, {len(todo)} to decode, "
          f"{len(bad)} excluded as undecodable")
    if not todo:
        return out

    model, processor = load_for_eval(model_id)
    try:
        with out.open("a", encoding="utf-8") as f:
            for start in tqdm(range(0, len(todo), BATCH_SIZE),
                              desc=f"{slug(model_id)}/{split}", unit="batch"):
                batch = todo[start:start + BATCH_SIZE]
                rows, chunks, owner = {}, [], []
                for i in batch:
                    row = d[i]                      # reads the audio bytes once
                    try:
                        pieces = chunk_audio(read_16k_mono(row["audio"]))
                    except Exception as e:
                        # Should be unreachable: cell 9 already decoded every row. Kept
                        # so one late failure cannot throw away an hour of decoding.
                        bad[row["segment_id"]] = f"{type(e).__name__}: {e}"
                        print(f"  skip {row['segment_id']}: {type(e).__name__}: {e}")
                        continue
                    rows[i] = row
                    for c in pieces:
                        chunks.append(c)
                        owner.append(i)
                if not chunks:
                    continue
                hyps = transcribe(model, processor, chunks)
                joined = {}
                for i, h in zip(owner, hyps):
                    joined.setdefault(i, []).append(h.strip())
                for i in rows:
                    row = rows[i]
                    f.write(json.dumps({
                        "segment_id": row["segment_id"],
                        "ref": row["segment_text"],
                        "hyp": " ".join(joined[i]).strip(),
                        "duration_seconds": row["duration_seconds"],
                        "n_chunks": len(joined[i]),
                        "cs_terms_count": row["cs_terms_count"],
                        "topic": row.get("topic"),
                    }, ensure_ascii=False) + "\n")
                f.flush()
    finally:
        free(model, processor)
    return out

def read_jsonl(model_id: str, split: str) -> list:
    with jsonl_path(model_id, split).open(encoding="utf-8") as f:
        return [json.loads(x) for x in f if x.strip()]

## Step 1 + 2 — reproduce the paper's base row before touching v5

The next cell decodes **base PhoWhisper-large on `test` only** (~1,615 segments). The one
after scores it under all four normalization variants and checks **G1**. If no variant
lands inside the tolerance, the paper comparison is off the table — the head-to-head still
runs, but no number from this notebook may be placed next to a number from Table 4.

In [ ]:
# Cell 13 -- STEP 1: base model on test. ~1,615 decodes.
decode_split(BASE_MODEL, "test")
print("rows:", len(read_jsonl(BASE_MODEL, "test")))

In [ ]:
# Cell 14 -- STEP 2 / GATE G1: which normalization reproduces 31.24 WER / 19.25 CER?
recs = read_jsonl(BASE_MODEL, "test")
target = PAPER["test"][BASE_MODEL]

print(f"paper base row (test): WER {target['wer']:.2f} | CER {target['cer']:.2f}")
print(f"tolerance: CER +/-{TOL_CER_PP} pp | WER +/-{TOL_WER_PP} pp\n")
print("| variant | CER % | dCER pp | WER % | dWER pp | ref word->digit | G1 |")
print("|---|---:|---:|---:|---:|---:|---|")

g1_hits = []
for name in VARIANTS:
    refs, hyps, sr, sh = norm_pair([r["ref"] for r in recs], [r["hyp"] for r in recs], name)
    s = score(refs, hyps)
    cer, wer = s["cer"] * 100, s["wer"] * 100
    dc, dw = cer - target["cer"], wer - target["wer"]
    ok = abs(dc) <= TOL_CER_PP and abs(dw) <= TOL_WER_PP
    if ok:
        g1_hits.append((abs(dc), name))
    print(f"| {name} | {cer:.2f} | {dc:+.2f} | {wer:.2f} | {dw:+.2f} | {sr.conversions} | "
          f"{'PASS' if ok else 'fail'} |")

if g1_hits:
    SCORING_VARIANT = sorted(g1_hits)[0][1]
    G1 = True
    print(f"\nG1 PASS -> scoring variant locked to: {SCORING_VARIANT}")
else:
    G1 = False
    print("\nG1 FAIL -- no variant reproduces the paper's base row.")
    print("Comparisons against Table 4 are NOT sound. The base-vs-v5 head-to-head stays")
    print(f"valid (identical normalization on both sides). Scoring variant: {SCORING_VARIANT}")

## Step 3 — the remaining three decodes

`v5` on `test`, then both models on `hard`. ~2,931 decodes; on a single T4 at
`BATCH_SIZE=8` greedy, expect roughly 1.5 h per model per full split. Resumable, so a dead
session does not restart from zero.

In [ ]:
# Cell 16 -- STEP 3: everything else.
for split in SPLITS:
    for model_id in MODELS:
        decode_split(model_id, split)

for split in SPLITS:
    for model_id in MODELS:
        print(f"{model_id:42s} {split:5s} {len(read_jsonl(model_id, split)):5d} rows")

In [ ]:
# Cell 17 -- STEP 4: the table. CER and WER, corpus-level, under SCORING_VARIANT.
# Paired bootstrap on the same segments in the same order -> GATE G2.
results, per_seg, dropped = {}, {}, {}
for split in SPLITS:
    # Score both models on the SAME segments: undecodable rows, or rows a half-finished
    # decode never reached, must not be in one column and absent from the other.
    by_model = {m: {r["segment_id"]: r for r in read_jsonl(m, split)} for m in MODELS}
    common = sorted(set.intersection(*(set(v) for v in by_model.values())))
    dropped[split] = {m: sorted(set(by_model[m]) - set(common)) for m in MODELS}
    n_drop = sum(len(v) for v in dropped[split].values())
    print(f"{split}: scoring {len(common)} shared segments"
          + (f" | {n_drop} row(s) present for only one model, excluded" if n_drop else ""))
    for model_id in MODELS:
        recs = [by_model[model_id][sid] for sid in common]     # same order on both sides
        refs, hyps, _, _ = norm_pair([r["ref"] for r in recs], [r["hyp"] for r in recs],
                                     SCORING_VARIANT)
        s = score(refs, hyps)
        ret = english_token_retention(refs, hyps)
        cc = [char_counts(r, h) for r, h in zip(refs, hyps)]
        wc = [word_counts(r, h) for r, h in zip(refs, hyps)]
        lo, hi = bootstrap_ci(cc)
        results[(model_id, split)] = {
            "n_segments": s["n_segments"], "cer": s["cer"], "wer": s["wer"],
            "cer_ci": [lo, hi], "retention": ret["retention"],
            "n_candidates": ret["n_candidates"],
        }
        per_seg[(model_id, split)] = {"char": cc, "word": wc,
                                      "ids": [r["segment_id"] for r in recs],
                                      "refs": refs, "hyps": hyps}

print(f"scoring variant: {SCORING_VARIANT} | G1: {'PASS' if G1 else 'FAIL'}\n")
print("| split | model | n | CER % | CER 95% CI | WER % | loanword retention |")
print("|---|---|---:|---:|---:|---:|---:|")
for split in SPLITS:
    for model_id in MODELS:
        r = results[(model_id, split)]
        ret = "-" if r["retention"] is None else f'{r["retention"]*100:.2f}% (n={r["n_candidates"]})'
        print(f'| {split} | `{model_id}` | {r["n_segments"]} | {r["cer"]*100:.2f} | '
              f'[{r["cer_ci"][0]*100:.2f}, {r["cer_ci"][1]*100:.2f}] | {r["wer"]*100:.2f} | {ret} |')

print("\n### GATE G2 -- paired bootstrap, positive delta = v5 better\n")
print("| split | dCER pp (base - v5) | 95% CI | verdict |")
print("|---|---:|---:|---|")
G2 = {}
for split in SPLITS:
    b, c = per_seg[(BASE_MODEL, split)], per_seg[(CAND_MODEL, split)]
    if b["ids"] != c["ids"]:
        raise ValueError(f"{split}: segment ids differ between models -- pairing is invalid")
    lo, hi = bootstrap_delta_ci(b["char"], c["char"])
    d = rate(b["char"]) - rate(c["char"])
    v = verdict(lo, hi)
    G2[split] = v == "IMPROVED"
    print(f"| {split} | {d*100:+.2f} | [{lo*100:+.2f}, {hi*100:+.2f}] | {v} |")

print("\n### GATE G3 -- SOTA claim\n")
for split in SPLITS:
    paper_cer = PAPER[split].get(BASE_MODEL, {}).get("cer")
    v5_cer = results[(CAND_MODEL, split)]["cer"] * 100
    if paper_cer is None:
        print(f"{split}: no paper baseline recorded (extract it from the PDF first) -- NO CLAIM")
    elif G1 and G2[split] and v5_cer < paper_cer:
        print(f"{split}: CLAIMABLE -- v5 CER {v5_cer:.2f} < paper CER {paper_cer:.2f}, G1+G2 pass")
    else:
        print(f"{split}: NO CLAIM -- v5 CER {v5_cer:.2f} vs paper {paper_cer:.2f} | "
              f"G1 {'pass' if G1 else 'FAIL'} | G2 {'pass' if G2[split] else 'FAIL'}")

payload = {
    "dataset": DATASET_ID, "scoring_variant": SCORING_VARIANT, "g1_reproduced": G1,
    "decode": {"num_beams": NUM_BEAMS, "language": LANGUAGE, "batch_size": BATCH_SIZE,
               "max_chunk_sec": MAX_CHUNK_SEC, "no_repeat_ngram_size": 0},
    "paper_table4": PAPER,
    "results": {f"{m}|{s}": results[(m, s)] for (m, s) in results},
    "g2_paired_improved": {s: G2[s] for s in SPLITS},
    "undecodable": {s: BAD[s] for s in SPLITS},
    "excluded_single_model": dropped,
}
(OUT_DIR / "scores.json").write_text(json.dumps(payload, ensure_ascii=False, indent=2),
                                     encoding="utf-8")
print("\nwrote", OUT_DIR / "scores.json")

## Step 5 — listen to the audio

Exports `N_SAMPLES` segments per split as 16 kHz wav, chosen **stratified by v5's
per-segment CER**: a third best, a third around the median, a third worst. The worst third
is where you hear whether a high CER is the model mishearing or the Gemini-generated
reference being wrong — ViMedCSS references were not transcribed by a human from scratch.

Each sample prints an inline player plus the reference and both hypotheses. Files also land
in `OUT_DIR/samples/<split>/` and in the zip, so they can be listened to after the session
ends.

In [ ]:
# Cell 19 -- audio samples for listening.
import soundfile as sf
from IPython.display import Audio, display, Markdown

def export_samples(split: str, n: int = N_SAMPLES):
    b, c = per_seg[(BASE_MODEL, split)], per_seg[(CAND_MODEL, split)]
    cers = [(cc.edits / cc.ref_len if cc.ref_len else 0.0) for cc in c["char"]]
    order = sorted(range(len(cers)), key=lambda i: cers[i])
    k = max(1, n // 3)
    mid = len(order) // 2
    picks = ([("best", i) for i in order[:k]]
             + [("median", i) for i in order[mid - k // 2: mid - k // 2 + k]]
             + [("worst", i) for i in order[-k:]])

    sample_dir = OUT_DIR / "samples" / split
    sample_dir.mkdir(parents=True, exist_ok=True)
    by_id = {sid: j for j, sid in enumerate(DS[split]["segment_id"])}
    lines = [f"# Samples -- {split} (scoring: {SCORING_VARIANT})"]

    display(Markdown(f"## {split}"))
    for band, i in picks:
        sid = c["ids"][i]
        row = DS[split][by_id[sid]]
        arr = read_16k_mono(row["audio"])
        wav = sample_dir / f"{band}_{sid}.wav"
        sf.write(wav, arr, TARGET_SR)

        bc = b["char"][i]
        base_cer = bc.edits / bc.ref_len if bc.ref_len else 0.0
        display(Markdown(f"**{band}** - `{sid}` - {row['duration_seconds']:.1f}s - "
                         f"CER v5 {cers[i]*100:.1f}% vs base {base_cer*100:.1f}% - "
                         f"CS terms {row['cs_terms_count']}"))
        display(Audio(str(wav)))
        display(Markdown(f"- ref  : {c['refs'][i]}\n"
                         f"- base : {b['hyps'][i]}\n"
                         f"- v5   : {c['hyps'][i]}"))
        lines += [f"\n## {band} - {sid} - {row['duration_seconds']:.1f}s",
                  f"- audio: `{wav.name}`",
                  f"- CER: v5 {cers[i]*100:.2f}% | base {base_cer*100:.2f}%",
                  f"- ref  : {c['refs'][i]}",
                  f"- base : {b['hyps'][i]}",
                  f"- v5   : {c['hyps'][i]}"]

    (sample_dir / "samples.md").write_text("\n".join(lines), encoding="utf-8")
    print(f"{split}: wrote {len(picks)} wav + samples.md to {sample_dir}")

for split in SPLITS:
    export_samples(split)

In [ ]:
# Cell 20 -- zip everything for download.
import shutil
zip_path = shutil.make_archive(str(OUT_DIR.parent / "vimedcss_output"), "zip", OUT_DIR)
print("download from the Kaggle output panel:", zip_path)